In [1]:
import random
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.options.display.max_columns = 100

import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import seaborn as sns
from collections import Counter
import json

import all_blocks
from merge_contracts import ContractsMerger
from generate_test import generate_preds
from get_distribution_utils import get_disrib_sums, cost_columns_to_datetime


c:\users\никита\appdata\local\programs\python\python39\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Подгружаем и обрабатываем датку

In [2]:
pays_df1 = pd.read_excel("data/Счета на оплату 3800-2023.XLSX")
pays_df2 = pd.read_excel("data/Счета на оплату 4200-4000-3800-2024.XLSX")
pays_df3 = pd.read_excel("data/Счета на оплату 5400-2023.XLSX")
pays_df4 = pd.read_excel("data/Счета на оплату 5400-2024.XLSX")
pays_df5 = pd.read_excel("data/Счета на оплату 5500-2023.XLSX")

In [3]:
pays_df = pd.concat([pays_df1, pays_df2, pays_df3, pays_df4, pays_df5], axis=0)

In [4]:
merger_df = pd.read_excel("data/Связь договор - здания.XLSX")
main_costs_df = pd.read_excel("data/Основные средства.XLSX")
squares = pd.read_excel("data/Площади зданий.XLSX")
serv_codes = pd.read_excel("data/Коды услуг.XLSX")



In [5]:
main_costs_df["тест_фича_нум"] = [random.randint(0, 100) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_бул"] = [random.randint(0, 1) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_дата"] = ["19.04.2006"] * len(main_costs_df)

In [6]:
merger = ContractsMerger(pays_df, merger_df, main_costs_df, squares, serv_codes)

E:\DIR_python_projects\ledaer_digital_transformation_24\zakupai\services\api\ml\lib\merge_contracts.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.needed_pays_df["time"] = self.needed_pays_df["Год"].map(lambda x: datetime(year=x + 1, month=1, day=1))


In [7]:
import time

t = time.time()
res, features = merger.start_merging()
print(time.time() - t)

57.62276363372803


In [8]:
# сохраняем результат с 1 страницы загрузки данных
res.to_csv("res_datetimes.csv")
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

res остается лежать на бэке где-то
features нужны на фронте чтобы блоки с признаками создать и на бэке дальше
то есть там всегда будут 3 блока условия + блоки действий + feature из этого файла

## Подгружаем граф, запускаем распределение

In [40]:
features = pickle.load(open("features.pkl", "rb"))
res = pd.read_csv("res_datetimes.csv", index_col=0)

C:\Users\843E~1\AppData\Local\Temp/ipykernel_13564/930177580.py:2: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  res = pd.read_csv("res_datetimes.csv", index_col=0)


In [41]:
res_by_prime.get_group("5006227284_1")

,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime,time,ID здания,Отношение действ. с,Отношение действ. до,Здание,Начало владения,Конец владения,Измер. действит. по,Измерение действ. с,Площадь,Единица измерения,ID основного средства,Класс основного средства,"Признак ""Используется в основной деятельности""","Признак ""Способ использования""",Площадь ОС,ЕИ площади,Дата начала действия связи с зданием,Дата окончания действия связи с зданием,Дата ввода в эксплуатацию,Дата выбытия,тест_фича_нум,тест_фича_бул,тест_фича_дата
10,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/337,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/337,1997-12-26,2040-01-01,9999-12-31 00:00:00,NaN,288.4,М2,38006040009270332,60401018,0,1,1.1,М2,1900-01-01,2040-01-01,1997-12-26,NaN,97,1,2006-04-19
11,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/337,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/337,1997-12-26,2040-01-01,9999-12-31 00:00:00,NaN,288.4,М2,38006040009270333,60401018,0,1,1.1,М2,1900-01-01,2040-01-01,1997-12-26,NaN,12,1,2006-04-19
12,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/341,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/341,1994-09-30,2040-01-01,9999-12-31 00:00:00,NaN,213.2,М2,38006190000019000,61908996,0,1,213.2,М2,1900-01-01,2040-01-01,1994-09-30,NaN,79,0,2006-04-19
13,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/360,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/360,1994-09-30,2040-01-01,9999-12-31 00:00:00,NaN,1922.5,М2,38006040007126002,60401018,1,1,1.1,М2,1900-01-01,2040-01-01,1994-09-30,NaN,45,1,2006-04-19
14,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/362,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/362,2011-07-12,2040-01-01,9999-12-31 00:00:00,NaN,506.0,М2,38006040007127101,60401018,1,1,1.0,М2,1900-01-01,2040-01-01,2011-07-12,2023-11-20 00:00:00,14,0,2006-04-19
15,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/366,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/366,1997-12-26,2040-01-01,9999-12-31 00:00:00,NaN,733.3,М2,38006040007125432,60401018,1,1,468.9,М2,2018-07-04,2040-01-01,1997-10-01,2023-10-24 00:00:00,63,0,2006-04-19
16,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/369,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/369,1997-10-01,2040-01-01,9999-12-31 00:00:00,NaN,265.5,М2,38006040007125442,60401018,1,1,1.1,М2,1900-01-01,2040-01-01,1997-10-01,NaN,97,0,2006-04-19
17,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/684,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/684,1996-08-28,2040-01-01,1998-02-18 00:00:00,1996-08-28 00:00:00,3259.3,М2,38006040008134291,60401018,0,0,0.0,М2,2017-02-03,2040-01-01,1997-12-31,2019-01-10 00:00:00,39,0,2006-04-19
18,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/684,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/684,1996-08-28,2040-01-01,1998-02-18 00:00:00,1996-08-28 00:00:00,3259.3,М2,38006040008134294,60401018,1,1,1.0,М2,1900-01-01,2040-01-01,1997-12-31,NaN,20,0,2006-04-19
19,3800,2023,5006227284,1,800003262,ДПН 3800/75785,44978,57231.97,5006227284_1,2024-01-01 00:00:00,ЗДН 3800/1/730,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/730,2013-07-23,2040-01-01,9999-12-31 00:00:00,NaN,261.1,М2,38006040014146121,60401018,0,0,0.0,М2,1900-01-01,2040-01-01,2013-05-30,NaN,78,1,2006-04-19


In [42]:
numeric_features = [i[0] for i in features if i[2] != "date"]
date_features = [i[0] for i in features if i[2] == "date"]

res = cost_columns_to_datetime(res, date_features)

In [43]:
res_by_prime = res.groupby("prime")

In [44]:
raw_graph = json.load(open("graphs/graph (5).json", "r"))

In [45]:
raw_graph["nodes"][2]["type"] = "use_feature_distribution"

In [46]:
all_blocks.init_graph(raw_graph, features)

In [47]:
unique_primes = res["prime"].unique()
distrib_sums = get_disrib_sums(res_by_prime, res["prime"].unique(), numeric_features)

100%|██████████| 4717/4717 [00:08<00:00, 548.67it/s]


In [48]:
res

,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime,time,ID здания,Отношение действ. с,Отношение действ. до,Здание,Начало владения,Конец владения,Измер. действит. по,Измерение действ. с,Площадь,Единица измерения,ID основного средства,Класс основного средства,"Признак ""Используется в основной деятельности""","Признак ""Способ использования""",Площадь ОС,ЕИ площади,Дата начала действия связи с зданием,Дата окончания действия связи с зданием,Дата ввода в эксплуатацию,Дата выбытия,тест_фича_нум,тест_фича_бул,тест_фича_дата
0,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.70,5006170938_1,2024-01-01 00:00:00,ЗДН 3800/1/265,2023-01-11 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/265,2007-11-26,2040-01-01,9999-12-31 00:00:00,NaN,616.3,М2,38006040010385572,60401018,0,1,1.10,М2,1900-01-01,2040-01-01,2007-11-26,NaN,90,0,2006-04-19
1,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.70,5006170938_1,2024-01-01 00:00:00,ЗДН 3800/1/1538,2023-01-11 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/1538,2018-08-10,2040-01-01,9999-12-31 00:00:00,2018-08-10 00:00:00,430.0,М2,38006080400345271,60804001,0,0,0.00,М2,1900-01-01,2040-01-01,2020-03-01,NaN,88,0,2006-04-19
2,3800,2023,5006205255,1,800001855,ДПН 3800/75304,NaN,65413.07,5006205255_1,2024-01-01 00:00:00,ЗДН 3800/1/360,2023-01-01 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/360,1994-09-30,2040-01-01,9999-12-31 00:00:00,NaN,1922.5,М2,38006040007126002,60401018,1,1,1.10,М2,1900-01-01,2040-01-01,1994-09-30,NaN,45,1,2006-04-19
3,3800,2023,5006205474,1,800006222,ДПН 3800/75274,44965,35405.00,5006205474_1,2024-01-01 00:00:00,ЗДН 3800/1/506,2023-01-01 00:00:00,2023-12-31 00:00:00,ЗДН 3800/1/506,1996-11-29,2040-01-01,9999-12-31 00:00:00,NaN,6274.8,М2,38006040009759672,60401018,1,1,1855.32,М2,2018-06-01,2040-01-01,1996-11-29,2024-02-05 00:00:00,89,1,2006-04-19
4,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/222,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/222,2007-09-03,2040-01-01,9999-12-31 00:00:00,NaN,236.9,М2,38006190000018780,61908996,0,1,236.90,М2,1900-01-01,2040-01-01,2013-05-30,NaN,20,0,2006-04-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101818,5500,2023,5006664663,1,800007337,ДПН 5500/183559,2023-12-27 00:00:00,23736.00,5006664663_1,2024-01-01 00:00:00,ЗДН 5500/1/3905,2023-01-30 00:00:00,2024-02-29 00:00:00,ЗДН 5500/1/3905,2023-01-30,2040-01-01,9999-12-31 00:00:00,2023-01-30 00:00:00,3848.6,М2,55006080400468400,60804001,1,0,3848.60,М2,1900-01-01,2040-01-01,2023-01-30,NaN,92,1,2006-04-19
101819,5500,2023,5006664663,1,800007337,ДПН 5500/183559,2023-12-27 00:00:00,23736.00,5006664663_1,2024-01-01 00:00:00,ЗДН 5500/1/3905,2023-01-30 00:00:00,2024-02-29 00:00:00,ЗДН 5500/1/3905,2023-01-30,2040-01-01,9999-12-31 00:00:00,2023-01-30 00:00:00,3848.6,М2,55006080400468401,60804001,0,0,0.00,М2,1900-01-01,2040-01-01,2023-01-30,NaN,37,1,2006-04-19
101820,5500,2023,5006664663,1,800007337,ДПН 5500/183559,2023-12-27 00:00:00,23736.00,5006664663_1,2024-01-01 00:00:00,ЗДН 5500/1/3914,2023-01-20 00:00:00,2024-02-29 00:00:00,ЗДН 5500/1/3914,2023-01-20,2040-01-01,9999-12-31 00:00:00,2023-01-20 00:00:00,88.6,М2,55006040054136050,60401018,1,0,88.60,М2,1900-01-01,2040-01-01,2008-07-04,NaN,53,0,2006-04-19
101821,5500,2023,5006664663,1,800007337,ДПН 5500/183559,2023-12-27 00:00:00,23736.00,5006664663_1,2024-01-01 00:00:00,ЗДН 5500/1/3959,2023-07-24 00:00:00,2024-02-29 00:00:00,ЗДН 5500/1/3959,2023-07-24,2040-01-01,9999-12-31 00:00:00,2023-07-24 00:00:00,1304.8,М2,55006040055290380,60401018,1,0,1304.80,М2,1900-01-01,2040-01-01,2012-12-29,NaN,13,0,2006-04-19


In [49]:
pays_by_prime = pays_df.groupby("prime")
res_by_prime = res.groupby("prime")

prime_id = "5006222329_1"
group_pay = pays_by_prime.get_group(prime_id)
group_res = res_by_prime.get_group(prime_id)

In [50]:
group_res

,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime,time,ID здания,Отношение действ. с,Отношение действ. до,Здание,Начало владения,Конец владения,Измер. действит. по,Измерение действ. с,Площадь,Единица измерения,ID основного средства,Класс основного средства,"Признак ""Используется в основной деятельности""","Признак ""Способ использования""",Площадь ОС,ЕИ площади,Дата начала действия связи с зданием,Дата окончания действия связи с зданием,Дата ввода в эксплуатацию,Дата выбытия,тест_фича_нум,тест_фича_бул,тест_фича_дата
4,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/222,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/222,2007-09-03,2040-01-01,9999-12-31 00:00:00,NaN,236.9,М2,38006190000018780,61908996,0,1,236.9,М2,1900-01-01,2040-01-01,2013-05-30,NaN,20,0,2006-04-19
5,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/578,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/578,1998-08-01,2040-01-01,9999-12-31 00:00:00,NaN,4027.8,М2,38006040007125691,60401018,0,0,0.0,М2,2017-11-03,2040-01-01,1998-08-01,2019-01-10 00:00:00,57,0,2006-04-19
6,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/579,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/579,1997-05-01,2040-01-01,9999-12-31 00:00:00,NaN,585.7,М2,38006040007125681,60401018,1,1,1.0,М2,1900-01-01,2040-01-01,1997-12-31,NaN,83,0,2006-04-19
7,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/664,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/664,1994-08-08,2040-01-01,9999-12-31 00:00:00,NaN,176.1,М2,38006040007123791,60401018,1,1,1.0,М2,1900-01-01,2040-01-01,1994-08-08,NaN,89,1,2006-04-19
8,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/667,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/667,1994-08-08,2040-01-01,9999-12-31 00:00:00,1994-08-08 00:00:00,222.3,М2,38006040010454893,60401018,1,1,1.1,М2,1900-01-01,2040-01-01,1994-08-08,NaN,100,1,2006-04-19
9,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1,2024-01-01 00:00:00,ЗДН 3800/1/668,2023-01-01 00:00:00,2024-12-31 00:00:00,ЗДН 3800/1/668,2006-02-15,2040-01-01,9999-12-31 00:00:00,NaN,7284.7,М2,38006040008059808,60401018,1,1,31.6,М2,2018-03-21,2040-01-01,1995-12-31,NaN,78,0,2006-04-19


In [51]:
group_pay

,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime
2257,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1


In [52]:
serv_codes_dct = {r["ID услуги"]: r["Класс услуги"] for _, r in serv_codes.iterrows()}

group_res = group_res.rename({"Год": "Год счета", "Дата отражения счета в учетной системе": "Дата отражения в учетной системе",
							  "ID здания": "Здание", "Класс основного средства": "Класс ОС", "ID услуги": "Услуга",
							  'Признак "Используется в основной деятельности"': 'Признак "Использование в основной деятельности"',
							  'Признак "Способ использования"': 'Признак "Способ использования"',}, axis=1)

group_res["Класс услуги"] = group_res["Услуга"].map(serv_codes_dct)
group_res["Сумма распределения"] = [i for i in distrib_sums[prime_id]]
group_res["Номер позиции распределения"] = [i for i in range(len(group_res))]

group_res = group_res[['Компания', 'Год счета', 'Номер счета', 'Позиция счета',
			   'Номер позиции распределения', 'Дата отражения в учетной системе',
			   'ID договора', 'Услуга', "Класс услуги", 'Здание', 'Класс ОС',
			   'ID основного средства', 'Признак "Использование в основной деятельности"',
			   'Признак "Способ использования"', 'Площадь ОС', 'Сумма распределения']]

In [56]:
group_res

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь ОС,Сумма распределения
4,3800,2023,5006222329,1,0,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/222,ЗДН 3800/1/222,61908996,38006190000018780,0,1,236.9,34897.690155
5,3800,2023,5006222329,1,1,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/578,ЗДН 3800/1/578,60401018,38006040007125691,0,0,0.0,0.000000
6,3800,2023,5006222329,1,2,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/579,ЗДН 3800/1/579,60401018,38006040007125681,1,1,1.0,147.309794
7,3800,2023,5006222329,1,3,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/664,ЗДН 3800/1/664,60401018,38006040007123791,1,1,1.0,147.309794
8,3800,2023,5006222329,1,4,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/667,ЗДН 3800/1/667,60401018,38006040010454893,1,1,1.1,162.040773
9,3800,2023,5006222329,1,5,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/668,ЗДН 3800/1/668,60401018,38006040008059808,1,1,31.6,4654.989485


In [57]:
group_res["Сумма распределения"].sum()

40009.340000000004

In [261]:
answer_ex = pd.read_csv("data/Шаблон ответа.csv", sep=";")

In [212]:
columns_list = ['Компания', 'Год счета', 'Номер счета', 'Позиция счета',
       'Номер позиции распределения', 'Дата отражения в учетной системе',
       'ID договора', 'Услуга', 'Класс услуги', 'Здание', 'Класс ОС',
       'ID основного средства',
       'Признак "Использование в основной деятельности"',
       'Признак "Способ использования"', 'Площадь', 'Сумма распределения',
       'Счет главной книги']

for i in columns_list:
	if i not in group_res.columns:
		print(i)

Счет главной книги


In [58]:
group_pay

,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime
2257,3800,2023,5006222329,1,800003262,ДПН 3800/75731,44977,40009.34,5006222329_1


In [59]:
group_res

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь ОС,Сумма распределения
4,3800,2023,5006222329,1,0,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/222,ЗДН 3800/1/222,61908996,38006190000018780,0,1,236.9,34897.690155
5,3800,2023,5006222329,1,1,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/578,ЗДН 3800/1/578,60401018,38006040007125691,0,0,0.0,0.000000
6,3800,2023,5006222329,1,2,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/579,ЗДН 3800/1/579,60401018,38006040007125681,1,1,1.0,147.309794
7,3800,2023,5006222329,1,3,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/664,ЗДН 3800/1/664,60401018,38006040007123791,1,1,1.0,147.309794
8,3800,2023,5006222329,1,4,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/667,ЗДН 3800/1/667,60401018,38006040010454893,1,1,1.1,162.040773
9,3800,2023,5006222329,1,5,44977,ДПН 3800/75731,800003262,S004,ЗДН 3800/1/668,ЗДН 3800/1/668,60401018,38006040008059808,1,1,31.6,4654.989485


In [70]:
group_res["Услуга"].map(serv_codes)

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [75]:
preds = generate_preds(pays_df, serv_codes, res, distrib_sums)

100%|██████████| 4717/4717 [00:08<00:00, 539.31it/s]


0        800003299
1        800003299
56190    800002266
56191    800002266
56192    800006222
           ...    
56185    800007377
56186    800007377
56187    800007377
56188    800000706
56189    800000706
Name: Услуга, Length: 101823, dtype: int64


In [74]:
preds

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь ОС,Сумма распределения,Счет главной книги,ID услуги
0,3800,2023,5006170938,1,0,2023-01-16 00:00:00,ДПН 3800/74661,800003299,S001,ЗДН 3800/1/265,60401018,38006040010385572,0,1,1.10,1.135427e+06,Unknown,S001
1,3800,2023,5006170938,1,1,2023-01-16 00:00:00,ДПН 3800/74661,800003299,S001,ЗДН 3800/1/1538,60804001,38006080400345271,0,0,0.00,0.000000e+00,Unknown,S001
56190,5400,2023,5006176256,1,0,2023-01-18 00:00:00,ДПН 5400/143136,800002266,S036,ЗДН 5400/1/7432,60804001,54006080400468090,1,0,150.90,6.955400e+04,Unknown,S036
56191,5400,2023,5006176256,2,0,2023-01-18 00:00:00,ДПН 5400/143136,800002266,S036,ЗДН 5400/1/7432,60804001,54006080400468090,1,0,150.90,5.155200e+04,Unknown,S036
56192,5400,2023,5006197559,1,0,2023-02-01 00:00:00,ДПН 5400/143743,800006222,S012,ЗДН 5400/1/6824,60804001,54006080400468690,1,0,971.09,1.179680e+04,Unknown,S012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56185,4200,2024,5006861096,1,82,2024-05-28 00:00:00,ДПН 4200/207590,800007377,S004,ЗДН 4200/1/6900,60401018,42006040054850570,1,0,5.53,1.457004e+01,Unknown,S004
56186,4200,2024,5006861096,1,83,2024-05-28 00:00:00,ДПН 4200/207590,800007377,S004,ЗДН 4200/1/7145,60804001,42006080400490040,1,0,30.70,8.088612e+01,Unknown,S004
56187,4200,2024,5006861096,1,84,2024-05-28 00:00:00,ДПН 4200/207590,800007377,S004,ЗДН 4200/1/7146,60804001,42006080400489420,1,0,11.25,2.964068e+01,Unknown,S004
56188,4200,2024,5006861117,1,0,2024-05-28 00:00:00,ДПН 4200/207349,800000706,S001,ЗДН 4200/1/423,60401018,42006040015620131,0,0,0.00,1.855949e+05,Unknown,S001


In [17]:
preds.to_csv("Результат_распределенные_счета.csv")